In [1]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text, inspect
#from sqlalchemy.exc import ProgrammingError
import pandas as pd
import gc

In [2]:
!pwd

/home/djmead/Documents/Projects/Financial/sql


In [3]:
load_dotenv()

True

In [4]:
USER = os.getenv("DB_USER")
PASSWORD = os.getenv("DB_PASSWORD")
HOST = os.getenv("DB_HOST")
PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

In [5]:
print(f"USER: {USER}")
print(f"PASSWORD: {PASSWORD[:1]}...{PASSWORD[-1:]}")
print(f"HOST: {HOST}")
print(f"PORT: {PORT}")
print(f"DB_NAME: {DB_NAME}")

USER: postgres
PASSWORD: C...9
HOST: localhost
PORT: 5432
DB_NAME: financial_db


In [ ]:
ADMIN_URL = f"postgresql+psycopg://{USER}:{PASSWORD}@{HOST}:{PORT}/{DB_NAME}"

admin_engine = create_engine(ADMIN_URL, isolation_level="AUTOCOMMIT")#, echo=True)

In [7]:
create_date_table_sql_path = 'CREATE_TABLE_dim_date.sql'

In [8]:
def run_sql_file(file_path):
    with open(file_path, 'r') as file:
        sql = file.read()
    with admin_engine.connect() as connection:
        connection.execute(text(sql))
        #connection.commit()
    print(f"Executed SQL file: {file_path}")

In [9]:
run_sql_file(create_date_table_sql_path)

Executed SQL file: CREATE_TABLE_dim_date.sql


In [ ]:
def describe_table(table_name, schema="public"):
    """Return column metadata for a PostgreSQL table."""
    from sqlalchemy import inspect

    inspector = inspect(admin_engine)
    columns = inspector.get_columns(table_name, schema=schema)
    primary_key_columns = inspector.get_pk_constraint(
        table_name, schema=schema
    ).get("constrained_columns", [])

    return [
        {
            "column_name": column["name"],
            "type": str(column["type"]),
            "nullable": column["nullable"],
            "default": column["default"],
            "primary_key": column["name"] in primary_key_columns,
        }
        for column in columns
    ]

describe_table("dim_date")

[{'column_name': 'id',
  'type': 'BIGINT',
  'nullable': False,
  'default': None,
  'primary_key': True},
 {'column_name': 'date',
  'type': 'TIMESTAMP',
  'nullable': False,
  'default': None,
  'primary_key': False},
 {'column_name': 'year',
  'type': 'INTEGER',
  'nullable': False,
  'default': None,
  'primary_key': False},
 {'column_name': 'quarter',
  'type': 'INTEGER',
  'nullable': False,
  'default': None,
  'primary_key': False},
 {'column_name': 'month',
  'type': 'INTEGER',
  'nullable': False,
  'default': None,
  'primary_key': False},
 {'column_name': 'day_of_month',
  'type': 'INTEGER',
  'nullable': False,
  'default': None,
  'primary_key': False},
 {'column_name': 'hour',
  'type': 'INTEGER',
  'nullable': False,
  'default': None,
  'primary_key': False},
 {'column_name': 'minute',
  'type': 'INTEGER',
  'nullable': False,
  'default': None,
  'primary_key': False}]

In [11]:
year = 1990
month = 1
print(f"{year:04d}-{month:02d}-01 00:00:00")


1990-01-01 00:00:00


In [12]:
year_month_list = []
for year in range(1990, 2030):
    for month in range(1, 13):
        year_month_list.append([year, month])
print(year_month_list)

[[1990, 1], [1990, 2], [1990, 3], [1990, 4], [1990, 5], [1990, 6], [1990, 7], [1990, 8], [1990, 9], [1990, 10], [1990, 11], [1990, 12], [1991, 1], [1991, 2], [1991, 3], [1991, 4], [1991, 5], [1991, 6], [1991, 7], [1991, 8], [1991, 9], [1991, 10], [1991, 11], [1991, 12], [1992, 1], [1992, 2], [1992, 3], [1992, 4], [1992, 5], [1992, 6], [1992, 7], [1992, 8], [1992, 9], [1992, 10], [1992, 11], [1992, 12], [1993, 1], [1993, 2], [1993, 3], [1993, 4], [1993, 5], [1993, 6], [1993, 7], [1993, 8], [1993, 9], [1993, 10], [1993, 11], [1993, 12], [1994, 1], [1994, 2], [1994, 3], [1994, 4], [1994, 5], [1994, 6], [1994, 7], [1994, 8], [1994, 9], [1994, 10], [1994, 11], [1994, 12], [1995, 1], [1995, 2], [1995, 3], [1995, 4], [1995, 5], [1995, 6], [1995, 7], [1995, 8], [1995, 9], [1995, 10], [1995, 11], [1995, 12], [1996, 1], [1996, 2], [1996, 3], [1996, 4], [1996, 5], [1996, 6], [1996, 7], [1996, 8], [1996, 9], [1996, 10], [1996, 11], [1996, 12], [1997, 1], [1997, 2], [1997, 3], [1997, 4], [1997, 5],

In [13]:
dates = pd.date_range(start="1990-01-01 00:00:00", end="2030-12-31 23:59:00", freq="min", tz="UTC")
dates

DatetimeIndex(['1990-01-01 00:00:00+00:00', '1990-01-01 00:01:00+00:00',
               '1990-01-01 00:02:00+00:00', '1990-01-01 00:03:00+00:00',
               '1990-01-01 00:04:00+00:00', '1990-01-01 00:05:00+00:00',
               '1990-01-01 00:06:00+00:00', '1990-01-01 00:07:00+00:00',
               '1990-01-01 00:08:00+00:00', '1990-01-01 00:09:00+00:00',
               ...
               '2030-12-31 23:50:00+00:00', '2030-12-31 23:51:00+00:00',
               '2030-12-31 23:52:00+00:00', '2030-12-31 23:53:00+00:00',
               '2030-12-31 23:54:00+00:00', '2030-12-31 23:55:00+00:00',
               '2030-12-31 23:56:00+00:00', '2030-12-31 23:57:00+00:00',
               '2030-12-31 23:58:00+00:00', '2030-12-31 23:59:00+00:00'],
              dtype='datetime64[us, UTC]', length=21564000, freq='min')

In [14]:
print(dates[0])
print(dates[-1])

1990-01-01 00:00:00+00:00
2030-12-31 23:59:00+00:00


In [15]:
df = pd.DataFrame()
df['Date'] = dates
df['year'] = dates.year
df['quarter'] = dates.quarter
df['month'] = dates.month
df['day_of_month'] = dates.day
df['hour'] = dates.hour
df['minute'] = dates.minute

del dates
gc.collect()

193

In [16]:
df = df[~df['Date'].duplicated()].reset_index(drop=True)


In [17]:
df['id'] = df.index

In [18]:
dft = df.iloc[0:3]

dft['Date'].astype(str).str.split("+", expand=True)[0]

0    1990-01-01 00:00:00
1    1990-01-01 00:01:00
2    1990-01-01 00:02:00
Name: 0, dtype: str

In [19]:
df.head()

,Date,year,quarter,month,day_of_month,hour,minute,id
0,1990-01-01 00:00:00+00:00,1990,1,1,1,0,0,0
1,1990-01-01 00:01:00+00:00,1990,1,1,1,0,1,1
2,1990-01-01 00:02:00+00:00,1990,1,1,1,0,2,2
3,1990-01-01 00:03:00+00:00,1990,1,1,1,0,3,3
4,1990-01-01 00:04:00+00:00,1990,1,1,1,0,4,4


In [20]:
df.tail()

,Date,year,quarter,month,day_of_month,hour,minute,id
21563995,2030-12-31 23:55:00+00:00,2030,4,12,31,23,55,21563995
21563996,2030-12-31 23:56:00+00:00,2030,4,12,31,23,56,21563996
21563997,2030-12-31 23:57:00+00:00,2030,4,12,31,23,57,21563997
21563998,2030-12-31 23:58:00+00:00,2030,4,12,31,23,58,21563998
21563999,2030-12-31 23:59:00+00:00,2030,4,12,31,23,59,21563999


In [ ]:
def write_dataframe_to_table(
    dataframe,
    table_name="dim_date",
    schema="public",
    engine=admin_engine,
    chunksize=8000,):
    """Append a dataframe to a SQL table and return the number of rows written."""
    #import pandas as pd
    #from sqlalchemy import inspect
    if dataframe.empty:
        return 0
    table_columns = [
        column["name"]
        for column in inspect(engine).get_columns(table_name, schema=schema)
    ]
    source = dataframe.copy()
    if "Date" in source.columns and "date" not in source.columns:
        source = source.rename(columns={"Date": "date"})
    if "date" not in source.columns:
        raise ValueError("The dataframe must contain a Date or date column")
    source["date"] = pd.to_datetime(source["date"])
    if source["date"].isna().any():
        raise ValueError("The date column contains null values")
    if "id" not in source.columns:
        source["id"] = source["date"].dt.strftime("%Y%m%d%H%M").astype("int64")
    if source["id"].duplicated().any():
        raise ValueError("The dataframe contains duplicate primary-key values")
    missing_columns = set(table_columns) - set(source.columns)
    if missing_columns:
        raise ValueError(
            f"The dataframe is missing required table columns: {sorted(missing_columns)}"
        )
    source = source[table_columns]
    with engine.begin() as connection:
        source.to_sql(
            table_name,
            con=connection,
            schema=schema,
            if_exists="append",
            index=False,
            chunksize=chunksize,
            method="multi",
        )
    return len(source)


# Run these lines to insert the complete dataframe:
# rows_written = write_dataframe_to_table(df)
# print(f"Inserted {rows_written:,} rows into public.dim_date")

In [22]:
#for year in pd.unique(df['year']):print(year)

In [23]:
#df[df['year'] == 2024]

In [24]:
def insert_by_year(df_in):
    years = pd.unique(df_in['year'])
    months = pd.unique(df_in['month'])
    for year in years:
        for month in months:
            print(f"Inserting rows for year {year}, month {month}")
            df_year_month = df_in[(df_in['year'] == year) & (df_in['month'] == month)]
            rows_written = write_dataframe_to_table(df_year_month)
            #print(f"Inserted {rows_written:,} rows into public.dim_date")
            del rows_written
            del df_year_month
            gc.collect()

In [25]:
df


,Date,year,quarter,month,day_of_month,hour,minute,id
0,1990-01-01 00:00:00+00:00,1990,1,1,1,0,0,0
1,1990-01-01 00:01:00+00:00,1990,1,1,1,0,1,1
2,1990-01-01 00:02:00+00:00,1990,1,1,1,0,2,2
3,1990-01-01 00:03:00+00:00,1990,1,1,1,0,3,3
4,1990-01-01 00:04:00+00:00,1990,1,1,1,0,4,4
...,...,...,...,...,...,...,...,...
21563995,2030-12-31 23:55:00+00:00,2030,4,12,31,23,55,21563995
21563996,2030-12-31 23:56:00+00:00,2030,4,12,31,23,56,21563996
21563997,2030-12-31 23:57:00+00:00,2030,4,12,31,23,57,21563997
21563998,2030-12-31 23:58:00+00:00,2030,4,12,31,23,58,21563998


In [ ]:
insert_by_year(df)

Inserting rows for year 1990, month 1
Inserting rows for year 1990, month 2
Inserting rows for year 1990, month 3
Inserting rows for year 1990, month 4
Inserting rows for year 1990, month 5
Inserting rows for year 1990, month 6
Inserting rows for year 1990, month 7
Inserting rows for year 1990, month 8
Inserting rows for year 1990, month 9
Inserting rows for year 1990, month 10
Inserting rows for year 1990, month 11
Inserting rows for year 1990, month 12
Inserting rows for year 1991, month 1
Inserting rows for year 1991, month 2
Inserting rows for year 1991, month 3
Inserting rows for year 1991, month 4
Inserting rows for year 1991, month 5
Inserting rows for year 1991, month 6
Inserting rows for year 1991, month 7


In [ ]:
df

In [ ]:
# Run these lines to insert the complete dataframe:
#rows_written = write_dataframe_to_table(df)
#print(f"Inserted {rows_written:,} rows into public.dim_date")